In [1]:
# ========================================
# 1. Install Dependencies
# ========================================

!pip install -qU \
    langchain \
    langchain-community \
    langchain-ollama \
    langchain-chroma \
    langchain-text-splitters \
    sentence-transformers \
    chromadb \
    pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/

In [2]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 163 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (734 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
!which ollama

/usr/local/bin/ollama


In [5]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(10)

print("Ollama server started!")

Ollama server started!


In [6]:
!curl http://localhost:11434

Ollama is running

In [7]:
!ollama pull gemma3:4b

In [8]:
!ollama list

NAME          ID              SIZE      MODIFIED               
gemma3:12b    f4031aab637d    8.1 GB    Less than a second ago    


In [9]:
import os

print(os.listdir("/content"))

['.config', 'sample_data']


In [11]:
import zipfile
import os

zip_path = "/content/archive (22).zip"
extract_path = "/content/recipes"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Files extracted successfully!")

Files extracted successfully!


In [12]:
for root, dirs, files in os.walk(extract_path):
    for file in files:
        if file.lower().endswith(".pdf"):
            print(os.path.join(root, file))

/content/recipes/Kadhi Pakora Dani Valent.pdf
/content/recipes/trimed-america-cook-book.pdf
/content/recipes/Paratha Global Young Academy.pdf
/content/recipes/Tarka Dal recipe TOPdesk Careers.pdf
/content/recipes/Murgh Pakora Crispy Indian Chicken Fritters Taz Doolittle.pdf
/content/recipes/indian Vegetarian instant pot cookbook Author Archana Mundhe.pdf
/content/recipes/Idli Recipes Sify Food.pdf
/content/recipes/Chana Masala Prana.pdf
/content/recipes/Aloo Gobi (Spicy Potato and Cauliflower) Bosch.pdf
/content/recipes/Dal Makhani Prana.pdf
/content/recipes/Thai_Recipes.pdf


In [13]:
# 3. Load PDFs

from langchain_community.document_loaders import (
    DirectoryLoader,
    PyPDFLoader
)

folder_path = "/content/recipes"
loader = DirectoryLoader(
    folder_path,
    glob="*.pdf",
    loader_cls=PyPDFLoader
)

data = loader.load()

print("Number of pages:", len(data))

/tmp/ipykernel_1232/2142873036.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


Number of pages: 223


In [14]:
# 4. Chunking

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

docs = text_splitter.split_documents(data)

print("Number of chunks:", len(docs))

Number of chunks: 384


In [15]:
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

class LocalEmbedding:

    def __init__(self):
        self.model = SentenceTransformer(
            "sentence-transformers/all-MiniLM-L6-v2"
        ).to(device)

    def embed_documents(self, docs):
        embeddings = self.model.encode(
            docs,
            convert_to_tensor=True,
            device=device,
            show_progress_bar=True
        )
        return embeddings.cpu().numpy().tolist()

    def embed_query(self, query):
        embedding = self.model.encode(
            query,
            convert_to_tensor=True,
            device=device
        )
        return embedding.cpu().numpy().tolist()

embeddings = LocalEmbedding()

print("Embedding model loaded!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!


In [16]:
from langchain_chroma import Chroma

vectorstoredb = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="recipe_collection_v2"
)

retriever = vectorstoredb.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10}
)

print("Vector database created!")

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Vector database created!


In [17]:
# 7. Initialize Gemma

from langchain_ollama import OllamaLLM

llm = OllamaLLM(
    model="gemma3:4b"
)

print("Gemma 3 loaded!")

Gemma 3 loaded!


**Multi Query**

In [21]:
# Multi Query RAG + Similarity Re-ranking

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.load import dumps, loads
from operator import itemgetter

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


# 1. Number of Query Variations

NUM_QUERY_VARIANTS = 3

# Number of final chunks sent to Gemma
TOP_N_FINAL = 5


# 2. Multi Query Prompt

template = """
You are an AI language model assistant.

Your task is to generate 3 different versions of the given user question
to retrieve relevant documents from a recipe vector database.

By generating multiple perspectives on the user question,
your goal is to overcome some of the limitations
of distance-based similarity search.

Generate 3 alternative questions.
Each question should express the same information need
using different wording.

Provide the alternative questions separated by newlines.

Original question:
{question}
"""

prompt_perspectives = ChatPromptTemplate.from_template(
    template
)


# 3. Generate 3 Query Variations

generate_queries = (
    prompt_perspectives
    | llm
    | StrOutputParser()
    | (lambda x: [
        line.strip()
        for line in x.split("\n")
        if line.strip()
    ])
)


# 4. Remove Duplicate Documents

def get_unique_union(documents: list[list]):

    flattened_docs = [
        dumps(doc)
        for sublist in documents
        for doc in sublist
    ]

    unique_docs = list(set(flattened_docs))

    return [
        loads(doc)
        for doc in unique_docs
    ]


# 5. Similarity Re-ranking

def rerank_documents(
    question,
    documents,
    top_n=TOP_N_FINAL
):

    # Embed original question
    query_embedding = embeddings.model.encode(
        question,
        convert_to_numpy=True
    )

    # Get document texts
    document_texts = [
        doc.page_content
        for doc in documents
    ]

    # Embed all retrieved documents
    document_embeddings = embeddings.model.encode(
        document_texts,
        convert_to_numpy=True,
        show_progress_bar=False
    )

    # Calculate cosine similarity
    scores = cosine_similarity(
        [query_embedding],
        document_embeddings
    )[0]

    # Combine documents with similarity scores
    ranked_results = sorted(
        zip(documents, scores),
        key=lambda x: x[1],
        reverse=True
    )

    # Select top N documents
    top_documents = [
        doc
        for doc, score in ranked_results[:top_n]
    ]

    print(
        f"\nSelected top {len(top_documents)} "
        f"documents out of {len(documents)}"
    )

    print("\nTop Similarity Scores:")

    for i, (doc, score) in enumerate(
        ranked_results[:top_n],
        1
    ):
        print(
            f"{i}. Similarity: {score:.4f}"
        )

    return top_documents


# 6. Multi Query Retrieval

def multi_query_retrieve(question):

    # Generate 3 Query Variations

    variants = generate_queries.invoke({
        "question": question
    })

    # Keep only 3 variations
    variants = variants[:NUM_QUERY_VARIANTS]

    # Original Question + 3 Variations
    all_queries = [
        question
    ] + variants

    print("\nQuery variants used:")

    for i, q in enumerate(
        all_queries,
        1
    ):
        print(f"{i}. {q}")


    # Retrieve Documents

    retrieved_per_query = [
        retriever.invoke(q)
        for q in all_queries
    ]

    print("\nRetrieved chunks:")

    for i, docs in enumerate(
        retrieved_per_query,
        1
    ):
        print(
            f"Query {i}: "
            f"{len(docs)} chunks"
        )


    # Remove Duplicate Documents

    unique_docs = get_unique_union(
        retrieved_per_query
    )

    print(
        "\nUnique chunks after "
        "deduplication:",
        len(unique_docs)
    )


    # Similarity Re-ranking

    top_docs = rerank_documents(
        question,
        unique_docs,
        top_n=TOP_N_FINAL
    )


    return top_docs


# 7. Final RAG Prompt

template = """
You are a helpful recipe assistant.

Answer the user's question using ONLY the provided recipe context.

If the answer is not available in the context,
say that the information was not found in the provided recipe books.

Do not invent ingredients, quantities, cooking times,
or preparation steps.

If multiple recipes match the user's request,
recommend the relevant recipes and explain briefly
why they match the user's requirements.

Context:
{context}

Question:
{question}
"""

prompt = ChatPromptTemplate.from_template(
    template
)


# 8. Final RAG Chain

final_rag_chain = (
    {
        "context": lambda x: multi_query_retrieve(
            x["question"]
        ),
        "question": itemgetter("question")
    }
    | prompt
    | llm
    | StrOutputParser()
)


# 9. Ask Question

question = """
I want a recipe that contains potatoes and cauliflower.
What recipes do you recommend?
"""

answer = final_rag_chain.invoke({
    "question": question
})


# 10. Display Results

print("\n" + "=" * 60)
print("QUESTION")
print("=" * 60)

print(question)


print("\n" + "=" * 60)
print("FINAL ANSWER")
print("=" * 60)

print(answer)


Query variants used:
1. 
I want a recipe that contains potatoes and cauliflower.
What recipes do you recommend?

2. Show me recipes with potato and cauliflower.
3. Find me cauliflower and potato dishes.
4. I'm looking for recipes using both potatoes and cauliflower – can you suggest some?

Retrieved chunks:
Query 1: 10 chunks
Query 2: 10 chunks
Query 3: 10 chunks
Query 4: 10 chunks

Unique chunks after deduplication: 40

Selected top 5 documents out of 40

Top Similarity Scores:
1. Similarity: 0.7238
2. Similarity: 0.7238
3. Similarity: 0.7238
4. Similarity: 0.7238
5. Similarity: 0.6182


/tmp/ipykernel_1232/1142192821.py:74: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  loads(doc)
/tmp/ipykernel_1232/1142192821.py:74: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  loads(doc)



QUESTION

I want a recipe that contains potatoes and cauliflower.
What recipes do you recommend?


FINAL ANSWER
I recommend the following recipes:

*   **Aloo Gobi (Spicy Potato and Cauliflower) Bosch:** This recipe explicitly states “Prepare vegetables by cutting potatoes into cubes and the cauliflower into florets.” It also includes steps for adding these vegetables to a pot with a paste.


**step back**

In [25]:
# Step-Back RAG

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.load import dumps, loads
from operator import itemgetter

from sklearn.metrics.pairwise import cosine_similarity


# 1. Step-Back Prompt

step_back_template = """
You are an AI assistant helping with recipe retrieval.

Given a specific user question, generate ONE more general
and abstract question that captures the underlying information need.

The step-back question should help retrieve broader and more
relevant recipe information from a recipe database.

Do not answer the question.
Only generate the step-back question.

Original question:
{question}

Step-back question:
"""

step_back_prompt = ChatPromptTemplate.from_template(
    step_back_template
)


# 2. Generate Step-Back Question

generate_step_back_query = (
    step_back_prompt
    | llm
    | StrOutputParser()
)


# 3. Step-Back Retrieval

def step_back_retrieve(question):

    # Generate general Step-Back question
    step_back_question = generate_step_back_query.invoke({
        "question": question
    }).strip()

    print("\nOriginal Question:")
    print(question)

    print("\nStep-Back Question:")
    print(step_back_question)


    # Retrieve using Original Question

    original_docs = retriever.invoke(
        question
    )


    # Retrieve using Step-Back Question

    step_back_docs = retriever.invoke(
        step_back_question
    )


    print("\nRetrieved chunks:")

    print(
        "Original Question:",
        len(original_docs),
        "chunks"
    )

    print(
        "Step-Back Question:",
        len(step_back_docs),
        "chunks"
    )


    # Combine Retrieved Documents

    retrieved_documents = [
        original_docs,
        step_back_docs
    ]


    # Remove Duplicates

    unique_docs = get_unique_union(
        retrieved_documents
    )

    print(
        "\nUnique chunks after deduplication:",
        len(unique_docs)
    )


    # Similarity Re-ranking

    top_docs = rerank_documents(
        question,
        unique_docs,
        top_n=TOP_N_FINAL
    )


    return top_docs

In [26]:
# 4. Final Step-Back RAG Prompt

final_template = """
You are a helpful recipe assistant.

Answer the user's question using ONLY the provided recipe context.

If the answer is not available in the context,
say that the information was not found in the provided recipe books.

Do not invent ingredients, quantities, cooking times,
or preparation steps.

If multiple recipes match the user's request,
recommend the relevant recipes and explain briefly
why they match the user's requirements.

Context:
{context}

Question:
{question}
"""

final_prompt = ChatPromptTemplate.from_template(
    final_template
)


# 5. Final Step-Back RAG Chain

step_back_rag_chain = (
    {
        "context": lambda x: step_back_retrieve(
            x["question"]
        ),
        "question": itemgetter("question")
    }
    | final_prompt
    | llm
    | StrOutputParser()
)

In [31]:
# Ask Question

question = """
I want a recipe that contains potatoes and cauliflower.
What recipes do you recommend?
"""

answer = step_back_rag_chain.invoke({
    "question": question
})


print("\n" + "=" * 60)
print("QUESTION")
print("=" * 60)

print(question)


print("\n" + "=" * 60)
print("FINAL ANSWER")
print("=" * 60)

print(answer)


Original Question:

I want a recipe that contains potatoes and cauliflower.
What recipes do you recommend?


Step-Back Question:
Suggest recipes featuring root vegetables and cruciferous vegetables.

Retrieved chunks:
Original Question: 10 chunks
Step-Back Question: 10 chunks

Unique chunks after deduplication: 20

Selected top 5 documents out of 20

Top Similarity Scores:
1. Similarity: 0.7238
2. Similarity: 0.7238
3. Similarity: 0.6182
4. Similarity: 0.6182
5. Similarity: 0.5719


/tmp/ipykernel_1232/1142192821.py:74: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  loads(doc)



QUESTION

I want a recipe that contains potatoes and cauliflower.
What recipes do you recommend?


FINAL ANSWER
I recommend the following recipes, as they both contain potatoes and cauliflower:

*   **Aloo Gobi (Spicy Potato and Cauliflower) Bosch:** This recipe includes instructions for preparing potatoes and cauliflower by cutting them into cubes and florets respectively. It then describes a method for sautéing these vegetables with a paste made from onions, ginger, turmeric, garlic, and cumin seeds.
*   **indian Vegetarian instant pot cookbook:** This recipe contains potato, cauliflower and rice as ingredients.
